In [2]:
import pandas as pd
from matplotlib.figure import Figure

In [9]:
old_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-09-14_90-90-90.xlsx', sheet_name='HHwork')
old_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-09-14_90-90-90.xlsx', sheet_name='EMPwork')
new_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-06_90-90-90.xlsx', sheet_name='HHwork')
new_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-06_90-90-90.xlsx', sheet_name='EMPwork')

In [10]:
from pathlib import Path

import numpy as np

RG_LABELS = {1: 'Metro', 2: 'Core Cities', 3: 'Larger Cities'}
# (capacity color, after-scaling color) per run
RUN_COLORS = {'old': ('lightsteelblue', 'tab:blue'), 'new': ('navajowhite', 'tab:orange')}


def _tod_shares(df, cap_col, run):
    """TOD rows of an HHwork/EMPwork table with start/final/increase columns suffixed by ``run``."""
    out = df[(df['is_tod'] == 1) & df['RGID'].isin(RG_LABELS)][
        ['nosplit_geo_id', 'RGID', 'name', cap_col, 'target_share']
    ].rename(columns={cap_col: f'start_{run}', 'target_share': f'final_{run}', 'RGID': f'RGID_{run}', 'name': f'name_{run}'})
    out[f'increase_{run}'] = out[f'final_{run}'] - out[f'start_{run}']
    return out


def plot_final_shares(old_df, new_df, cap_col, title, output_dir=None, file_name=None):
    """Plot starting (capacity) and final TOD shares per TOD area for the old and new runs.

    Same layout as ``plot_final_shares`` in ``split_ct_to_hct.py``: stacked bars
    with the capacity share below and the change to the final share on top,
    one panel per regional geography stacked vertically. Old and new runs are drawn side by side
    for each TOD area in different colors.

    Args:
        old_df, new_df: HHwork/EMPwork tables with ``is_tod``, ``RGID``,
            ``nosplit_geo_id``, ``name``, ``cap_col`` and ``target_share`` columns.
        cap_col: Column holding the starting capacity share (``DUcapshare`` or ``EMPcapshare``).
        title: Figure title.
        output_dir: If given, the image is saved here (created if missing).
        file_name: Image file name; required when ``output_dir`` is given.

    Returns:
        The matplotlib Figure, or None if there was nothing to plot.
    """
    bars = _tod_shares(old_df, cap_col, 'old').merge(
        _tod_shares(new_df, cap_col, 'new'), on='nosplit_geo_id', how='outer'
    )
    if bars.empty:
        return None
    bars['RGID'] = bars['RGID_new'].fillna(bars['RGID_old'])
    bars['name'] = bars['name_new'].fillna(bars['name_old'])
    bars['sort'] = bars['start_new'].fillna(bars['start_old'])

    rg_ids = [rgid for rgid in RG_LABELS if (bars['RGID'] == rgid).any()]
    widths = [(bars['RGID'] == rgid).sum() for rgid in rg_ids]
    fig = Figure(figsize=(max(8, 0.4 * max(widths) + 2), 6 * len(rg_ids)), layout='constrained')
    axes = fig.subplots(len(rg_ids), 1, squeeze=False, sharey=True)
    bar_w = 0.4
    for ax, rgid in zip(axes[:, 0], rg_ids):
        sub = bars[bars['RGID'] == rgid].sort_values('sort', ascending=False)
        x = np.arange(len(sub))
        for run, offset in [('old', -bar_w / 2), ('new', bar_w / 2)]:
            cap_color, scaled_color = RUN_COLORS[run]
            ax.bar(x + offset, sub[f'start_{run}'], width=bar_w, color=cap_color, label=f'{run} capacity')
            ax.bar(x + offset, sub[f'increase_{run}'], width=bar_w, bottom=sub[f'start_{run}'],
                   color=scaled_color, label=f'{run} after scaling')
        ax.set_xticks(x)
        ax.set_xlim(-0.5, max(widths) - 0.5)  # same bar width in every panel
        ax.set_xticklabels(sub['name'], rotation=90, fontsize=7)
        ax.set_title(RG_LABELS[rgid], fontsize=9)
        ax.set_ylim(0, 100)
        ax.set_ylabel('HCT shares (%)')
    axes[0][0].legend(title='HCT shares', loc='upper right')
    fig.suptitle(title)

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_dir / file_name, dpi=150)
    return fig


# Set to a directory (e.g. Path('output/plots')) to also save the images
PLOT_DIR = Path('output/plots')

figs = [
    plot_final_shares(old_hh, new_hh, 'DUcapshare', 'Households', PLOT_DIR, 'hct_final_shares_HH_comparison.png'),
    plot_final_shares(old_emp, new_emp, 'EMPcapshare', 'Employment', PLOT_DIR, 'hct_final_shares_Emp_comparison.png'),
]
for fig in figs:
    display(fig)

<Figure size 1560x1800 with 3 Axes>

<Figure size 1560x1800 with 3 Axes>